# Stage 6d -- Diagnosis Inference from Symptom Clusters

**Input** : Stage 6 `symptom_relations.json` (clusters) + Stage 5 `routed_terms.json` (grounded concepts + evidence quotes)
**Output**: `patient_records/<patient>/admissions/<hadm>/stage_06d_diagnosis_inference/inferred_diagnoses.json`

## Why this stage exists

Until now the pipeline went: symptoms -> SNOMED concepts -> clusters -> *(nothing)* -> ICD codes.
Stage 6c was mapping **raw symptoms** to ICD-10, and the chapter distribution shows exactly what
that produces:

| ICD-10 chapter | Ground truth | Stage 6c (mapping symptoms) |
|---|---|---|
| **R** = symptoms/signs, *not elsewhere classified* | 5.7% | **33.7%** |
| **Z** = factors influencing health status | 17.4% | 0% |
| I = circulatory | 19.0% | 8.9% |
| C = neoplasms | 5.3% | 1.0% |

Mapping symptoms yields *symptom* codes. Real discharge coding assigns the **disease**, using
R-codes only when no definitive diagnosis is established (5.7% of the time here). The missing
step is the one a clinician actually performs: reading a set of co-occurring findings and
naming the condition underlying them.

That is what Stage 6's clustering was built for -- "do these genuinely different symptoms
converge on one diagnosis?" -- but nothing consumed the clusters. This stage does.

```
Stage 5  symptoms  -> SNOMED concepts
Stage 6  concepts  -> clusters
Stage 6d clusters  -> candidate DIAGNOSES     <- THIS NOTEBOOK
Stage 6c diagnoses -> ICD-10 codes
Stage 7  6b + 6c   -> final decision
```

**Run order note**: despite the lettering, this runs *before* 6c. The suffix reflects when it
was written, not pipeline position -- 6c should be pointed at this stage's output instead of
raw symptoms.

## Method, and what it deliberately does not use

Each cluster is passed to the same local LLM the pipeline already uses (qwen2.5:7b via Ollama,
via `pipeline.call_llm_json`) together with the **verbatim evidence quotes** for its symptoms,
not just the term names -- the wording in the note carries clinical detail that a normalized
term drops.

The prompt is deliberately conservative. The model is asked to name only conditions the listed
findings actually support, to return an empty list when they don't converge on anything
specific, and explicitly *not* to pad with plausible-sounding comorbidities. Over-generation
here is costly: every invented diagnosis becomes ICD candidates downstream, diluting precision.

**Prior-admission codes are deliberately withheld from the prompt.** Stage 6b already supplies
those, and feeding them in would let the model simply restate the patient's history back --
making this stage's contribution unmeasurable and double-counting the same evidence in Stage 7.
Keeping 6d symptom-driven means its output is genuinely independent of 6b, which is what makes
their agreement informative rather than circular.

Each inferred diagnosis is then grounded to SNOMED via the same `ground_term()` logic Stage 5
uses (semantic-tag filtered to disorder/finding, cosine-reranked), so downstream mapping starts
from a real concept rather than free text.


## 1. Setup

In [ ]:
import json
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

from snomed_ontology import (
    configure,
    configure_bioportal,
    load_umls_api_key,
    load_bioportal_api_key,
    search_snomed,
    search_snomed_bioportal,
    get_sctid,
    get_cui_for_sctid,
    get_semantic_tag,
    GROUNDABLE_SEMANTIC_TAGS,
    embed,
    cosine_sim,
    MAX_WORKERS,
)
from pipeline import (
    LLMNotAvailableError,
    call_llm_json,
    check_llm,
    get_llm_config,
    print_pipeline_banner,
)

PROJECT_ROOT     = NB_DIR.parent
RECORDS_DIR      = PROJECT_ROOT / "patient_records"
STAGE_05_OUTPUT  = "stage_05_ontology_routing_agent"
STAGE_06_OUTPUT  = "stage_06_cross_symptom_routing"
STAGE_6D_OUTPUT  = "stage_06d_diagnosis_inference"

configure(load_umls_api_key(PROJECT_ROOT))
configure_bioportal(load_bioportal_api_key(PROJECT_ROOT))

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
print(f"LLM ready for diagnosis inference -- {LLM_CONFIG.method_prefix()}: {model_info}")

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith("patient_")])
print(f"Patients found: {len(patients)}")


## 2. Assemble each cluster with its evidence

A cluster from Stage 6 is just a list of term strings. To reason about it clinically the model
needs the grounded concept name *and* the verbatim quote the term was extracted from, so both
are pulled back out of Stage 5's output.


In [ ]:
def load_symptom_details(routed_terms: dict) -> dict:
    """{term: {concept_name, sctid, evidence}} for every grounded CURRENT_SYMPTOMS term."""
    details = {}
    for branch in routed_terms.get("routed_branches", []):
        for s in branch.get("symptoms", []):
            routing = s.get("routing")
            if routing and routing.get("grounded"):
                g = routing["grounded"]
                details[s["term"]] = {
                    "concept_name": g.get("name", ""),
                    "sctid": g.get("sctid"),
                    "evidence": s.get("evidence", ""),
                }
    return details


def build_cluster_payloads(relations: dict, details: dict) -> list:
    """Pair each Stage 6 cluster with its members' concept names and evidence quotes."""
    payloads = []
    for cluster in relations.get("clusters", []):
        members = []
        for term in cluster["terms"]:
            d = details.get(term, {})
            members.append({
                "term": term,
                "concept_name": d.get("concept_name", term),
                "evidence": d.get("evidence", ""),
            })
        payloads.append({"size": cluster["size"], "members": members})
    return payloads


## 3. The inference prompt

Two rules do most of the work here:

- **Empty is a valid answer.** A single non-specific finding (isolated nausea, malaise) often
  supports no particular diagnosis, and saying so is more useful than guessing.
- **No padding.** The model is told not to add likely-sounding comorbidities that the listed
  findings don't actually evidence, because each one becomes ICD candidates downstream.

`confidence` is the model's own stated certainty and is carried through to Stage 7 as one input
among several -- not treated as calibrated probability.


In [ ]:
DIAGNOSIS_SYSTEM_PROMPT = """You are a clinician reviewing findings documented during a single \
hospital admission. Given a group of co-occurring clinical findings (with the verbatim text they \
were extracted from), name the underlying condition(s) these findings support.

Rules:
- Name the DISEASE or CONDITION, not the finding. "Hepatic encephalopathy" and "liver failure" \
together support "Hepatic failure" or "Cirrhosis"; do not simply restate the findings.
- Only name conditions the listed findings actually support. Do NOT add likely comorbidities, \
risk factors, or conditions this patient might plausibly also have.
- If the findings do not converge on any specific condition (e.g. a single non-specific symptom), \
return an empty list. This is a valid and useful answer.
- At most 3 diagnoses per group, ordered most to least supported.
- Use standard clinical terminology, the kind that would appear as a SNOMED CT or ICD-10 term.

Return ONLY valid JSON:
{
  "diagnoses": [
    {"diagnosis": "standard clinical name", "confidence": 0.0-1.0, "reasoning": "one short phrase"}
  ]
}"""


def infer_diagnoses(cluster_payload: dict, config) -> list:
    """Ask the LLM what condition(s) a cluster of findings supports."""
    lines = []
    for m in cluster_payload["members"]:
        quote = f' (documented as: "{m["evidence"]}")' if m["evidence"] else ""
        lines.append(f'- {m["concept_name"]}{quote}')
    user_prompt = "Findings documented in this admission:\n" + "\n".join(lines)

    try:
        result = call_llm_json(DIAGNOSIS_SYSTEM_PROMPT, user_prompt, config)
    except (LLMNotAvailableError, ValueError) as e:
        return [{"diagnosis": "", "confidence": 0.0, "reasoning": f"inference failed: {e}", "error": True}]

    out = []
    for d in (result.get("diagnoses") or [])[:3]:
        name = str(d.get("diagnosis", "")).strip()
        if not name:
            continue
        try:
            conf = float(d.get("confidence", 0.0))
        except (TypeError, ValueError):
            conf = 0.0
        out.append({
            "diagnosis": name,
            "confidence": round(max(0.0, min(1.0, conf)), 3),
            "reasoning": str(d.get("reasoning", ""))[:200],
        })
    return out


## 4. Ground each inferred diagnosis to SNOMED

Same approach as Stage 5's `ground_term()`: pool UMLS + BioPortal candidates, deduplicate by
resolved SCTID (not by name -- two different concepts can share a display string), keep only
`disorder`/`finding` semantic tags, then pick by cosine similarity to the diagnosis name.


In [ ]:
def ground_diagnosis(name: str):
    """Resolve an inferred diagnosis name to a SNOMED CT concept, or None."""
    raw_pool = list(search_snomed_bioportal(name)) + list(search_snomed(name))
    if not raw_pool:
        return None

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        sctids = list(executor.map(lambda c: c.get("sctid") or get_sctid(c.get("cui", "")), raw_pool))
        pool, seen = [], set()
        for c, sctid in zip(raw_pool, sctids):
            if sctid and sctid not in seen:
                seen.add(sctid)
                pool.append({**c, "sctid": sctid})
        if not pool:
            return None
        tags = list(executor.map(lambda c: get_semantic_tag(c["sctid"]), pool))

    groundable = [c for c, tag in zip(pool, tags) if tag in GROUNDABLE_SEMANTIC_TAGS]
    if groundable:
        pool = groundable

    if len(pool) == 1:
        picked = pool[0]
    else:
        try:
            qvec = embed(name)
            picked = max(pool, key=lambda c: cosine_sim(qvec, embed(c["name"])))
        except requests.RequestException:
            picked = pool[0]

    sctid = picked.get("sctid")
    if not sctid:
        return None
    return {
        "cui": picked.get("cui") or get_cui_for_sctid(sctid),
        "sctid": sctid,
        "name": picked["name"],
    }


## 5. Test on one admission before the full batch

In [ ]:
test_adm = RECORDS_DIR / "patient_17774110" / "admissions" / "hadm_27339772"
with open(test_adm / STAGE_05_OUTPUT / "routed_terms.json", encoding="utf-8") as f:
    _routed = json.load(f)
with open(test_adm / STAGE_06_OUTPUT / "symptom_relations.json", encoding="utf-8") as f:
    _relations = json.load(f)

_details = load_symptom_details(_routed)
_payloads = build_cluster_payloads(_relations, _details)
print(f"{len(_payloads)} clusters in this admission\n")

for _p in _payloads[:4]:
    _names = [m["concept_name"] for m in _p["members"]]
    print(f'CLUSTER (size {_p["size"]}): {_names}')
    for _d in infer_diagnoses(_p, LLM_CONFIG):
        _g = ground_diagnosis(_d["diagnosis"]) if _d["diagnosis"] else None
        print(f'   -> {_d["diagnosis"]!r} (conf {_d["confidence"]}) -- {_d["reasoning"]}')
        print(f'      grounded: {_g}')
    print()


## 6. Run across all patients

In [ ]:
all_inferences = []

for patient_dir in patients:
    patient_id = patient_dir.name.replace("patient_", "")
    adm_root = patient_dir / "admissions"
    adm_dirs = sorted(adm_root.iterdir()) if adm_root.exists() else []

    for adm_dir in adm_dirs:
        routed_path    = adm_dir / STAGE_05_OUTPUT / "routed_terms.json"
        relations_path = adm_dir / STAGE_06_OUTPUT / "symptom_relations.json"
        if not routed_path.exists():
            print(f"  SKIP {patient_id}/{adm_dir.name} -- no Stage 5 output")
            continue

        with open(routed_path, encoding="utf-8") as f:
            routed = json.load(f)
        relations = {}
        if relations_path.exists():
            with open(relations_path, encoding="utf-8") as f:
                relations = json.load(f)

        details = load_symptom_details(routed)
        if relations.get("clusters"):
            payloads = build_cluster_payloads(relations, details)
        else:
            # Fewer than 2 grounded symptoms means Stage 6 produced no clusters --
            # fall back to treating each grounded symptom as its own singleton group
            # rather than skipping the admission entirely.
            payloads = [{"size": 1, "members": [{"term": t, **d}]} for t, d in details.items()]

        clusters_out = []
        for payload in payloads:
            try:
                diagnoses = infer_diagnoses(payload, LLM_CONFIG)
            except Exception as e:
                print(f"  ERROR inferring for {patient_id}/{adm_dir.name}: {e}")
                diagnoses = []

            for d in diagnoses:
                d["grounded"] = ground_diagnosis(d["diagnosis"]) if d.get("diagnosis") else None

            clusters_out.append({
                "cluster_terms": [m["term"] for m in payload["members"]],
                "cluster_size": payload["size"],
                "inferred_diagnoses": diagnoses,
            })

        n_dx = sum(len(c["inferred_diagnoses"]) for c in clusters_out)
        n_grounded = sum(1 for c in clusters_out for d in c["inferred_diagnoses"] if d.get("grounded"))

        output = {
            "patient_id": patient_id,
            "admission_id": adm_dir.name.replace("hadm_", ""),
            "n_clusters": len(clusters_out),
            "n_inferred_diagnoses": n_dx,
            "n_grounded_diagnoses": n_grounded,
            "clusters": clusters_out,
        }

        out_dir = adm_dir / STAGE_6D_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / "inferred_diagnoses.json", "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2)

        all_inferences.append(output)
        print(f"Patient {patient_id} | {adm_dir.name} | {len(clusters_out)} clusters -> "
              f"{n_dx} diagnoses ({n_grounded} grounded)")

print(f"\nDone. {len(all_inferences)} admissions processed.")


## 7. Inspect one admission

In [ ]:
EXAMPLE_IDX = 11
ex = all_inferences[EXAMPLE_IDX]

print(f'Patient {ex["patient_id"]} | Admission {ex["admission_id"]}')
print(f'{ex["n_clusters"]} clusters -> {ex["n_inferred_diagnoses"]} diagnoses '
      f'({ex["n_grounded_diagnoses"]} grounded to SNOMED)')
print()
for c in ex["clusters"]:
    print(f'CLUSTER: {c["cluster_terms"]}')
    if not c["inferred_diagnoses"]:
        print("   -> (no diagnosis inferred)")
    for d in c["inferred_diagnoses"]:
        g = d.get("grounded")
        print(f'   -> {d["diagnosis"]!r} (conf {d["confidence"]})')
        print(f'      {d["reasoning"]}')
        print(f'      grounded: {g["name"] + " [" + g["sctid"] + "]" if g else "NOT GROUNDED"}')
    print()


## What changes downstream

Stage 6c should now map **these inferred diagnoses** rather than raw symptoms. The expected
effect, given the chapter analysis at the top: fewer R-chapter symptom codes, more disease-chapter
codes, which is what the ground truth is actually made of.

Two things to watch when this feeds 6c:

- **Over-generation.** If the model names diagnoses the findings don't support, precision drops
  and the "no padding" instruction is not working. Compare `n_inferred_diagnoses` against
  `n_clusters` -- clusters averaging close to 3 diagnoses each is a warning sign.
- **Grounding failures.** `n_grounded_diagnoses` well below `n_inferred_diagnoses` means the
  model is producing names SNOMED does not recognize, which usually indicates non-standard
  phrasing rather than a grounding bug.
